<a href="https://colab.research.google.com/github/uznrbegum/autograders/blob/main/multitabledatabase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import csv #içeri aktardık gerekli olanları

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
conn = sqlite3.connect('tracks.sqlite')
cur = conn.cursor()

# Önce var olan tabloları temizle
cur.executescript('''
DROP TABLE IF EXISTS Artist;
DROP TABLE IF EXISTS Genre;
DROP TABLE IF EXISTS Album;
DROP TABLE IF EXISTS Track;
''')

# Tabloları oluştur
cur.executescript('''
CREATE TABLE Artist (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name    TEXT UNIQUE
);

CREATE TABLE Genre (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name    TEXT UNIQUE
);

CREATE TABLE Album (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    artist_id  INTEGER,
    title   TEXT UNIQUE
);

CREATE TABLE Track (
    id  INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    title TEXT  UNIQUE,
    album_id  INTEGER,
    genre_id  INTEGER,
    len INTEGER, rating INTEGER, count INTEGER
);
''')


In [4]:
filename = 'tracks.csv'  # Yüklediğin dosya adı

with open(filename, newline='', encoding='utf-8') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)  # Başlıktan kurtul

    for row in reader:
        if len(row) < 7:
            continue

        name = row[0]
        artist = row[1]
        album = row[2]
        genre = row[3]
        length = row[4]
        rating = row[5]
        count = row[6]

        # Sanatçıyı ekle
        cur.execute('INSERT OR IGNORE INTO Artist (name) VALUES (?)', (artist,))
        cur.execute('SELECT id FROM Artist WHERE name = ?', (artist,))
        artist_id = cur.fetchone()[0]

        # Albümü ekle
        cur.execute('INSERT OR IGNORE INTO Album (title, artist_id) VALUES (?, ?)', (album, artist_id))
        cur.execute('SELECT id FROM Album WHERE title = ?', (album,))
        album_id = cur.fetchone()[0]

        # Türü ekle
        cur.execute('INSERT OR IGNORE INTO Genre (name) VALUES (?)', (genre,))
        cur.execute('SELECT id FROM Genre WHERE name = ?', (genre,))
        genre_id = cur.fetchone()[0]

        # Şarkıyı ekle
        cur.execute('''INSERT OR REPLACE INTO Track
            (title, album_id, genre_id, len, rating, count)
            VALUES (?, ?, ?, ?, ?, ?)''',
            (name, album_id, genre_id, length, rating, count))

conn.commit()


In [5]:
sqlstr = '''
SELECT Track.title, Artist.name, Album.title, Genre.name
FROM Track
JOIN Genre ON Track.genre_id = Genre.id
JOIN Album ON Track.album_id = Album.id
JOIN Artist ON Album.artist_id = Artist.id
ORDER BY Artist.name LIMIT 3
'''

for row in cur.execute(sqlstr):
    print(row)


('For Those About To Rock (We Salute You)', 'AC/DC', 'Who Made Who', '84')
('Hells Bells', 'AC/DC', 'Who Made Who', '82')
('Shake Your Foundations', 'AC/DC', 'Who Made Who', '85')


In [6]:
from google.colab import files
files.download('tracks.sqlite')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>